In [ ]:
""" --- Integration test ---
 Author: Fuad Godzhaev
 Integration of multiple functions we have so far and testing if they work together
 The step-by-step process is as follows:
 1) Pre-processing: denoising, rotation, contrast adjustments
 2) Text-detection: segmentation of areas containing text
 3) Character recognition: breaking down highlited text region into individual characters for recognition

 TODO:
 -Check for colored/grey images; implement different behaviour
 -More robust edges and contour detection
 -Check for the image needing rotation (i.e. img3)
"""

import cv2
import numpy as np
import easygui as eg
import easyocr as ocr
from matplotlib import pyplot as plt

# 2. Pre-processing
def pre_processing(img):

    l, a, b = cv2.split(cv2.cvtColor(img, cv2.COLOR_RGB2LAB))

    bg = cv2.GaussianBlur(l, (51,51), 0)

    subtracted = cv2.subtract(l, bg)
    normalized = cv2.normalize(subtracted, None, 0, 255, cv2.NORM_MINMAX)

    # Applying denoising filter
    denoised = cv2.fastNlMeansDenoising(normalized, None, 10, 7, 21)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l_enh = clahe.apply(denoised)

    smoothed = cv2.GaussianBlur(l_enh, (3,3), 0)
    
    # Adaptive thresholding for local contrast handling, works better than global threshold for uneven lighting
    thresh = cv2.adaptiveThreshold(smoothed, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 11)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    opening = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel, 2)
    
    return(text_detection(l, opening))

# 3. Text detection
def text_detection(Gray, opening):
    # Invert so signatures are white on black background for contour detection
    opening_inv = cv2.bitwise_not(opening)

    # Find contours with hierarchy to identify both outer contours and holes within signatures
    contours, hierarchy = cv2.findContours(opening_inv, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)

    # Create empty mask
    mask = np.zeros_like(opening_inv)

    # Filter contours based on geometric properties to identify signatures
    for i, contour in enumerate(contours):

        # Skip holes (child contours), only process parent contours
        if hierarchy[0][i][3] != -1:
            continue

        # Filter by area: too small = noise, too large = not text
        area = cv2.contourArea(contour)
        if area < 100 or area > 500000:
            continue
            
        # Get bounding box for aspect ratio and extent calculations
        x, y, w, h = cv2.boundingRect(contour)
        # Avoid division by 0
        if h == 0:
            continue

        aspect_ratio = w / float(h)
        if aspect_ratio < 1 or aspect_ratio > 25.0:
            continue

        # Calculate convex hull and solidity to filter very irregular shapes
        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)
        if hull_area == 0:
            continue
    
        # Filter very solid/convex shapes
        solidity = area / hull_area    
        if solidity > 0.98:
            continue

        # Draw the signature contour as white on the mask
        cv2.drawContours(mask, [contour], 0, 255, -1)

        # Draw all child contours (holes within signatures) as black to preserve them
        child_idx = hierarchy[0][i][2]
        while child_idx != -1:
            cv2.drawContours(mask, [contours[child_idx]], 0, 0, -1)
            child_idx = hierarchy[0][child_idx][0]

    
    # Fill small gaps in signature strokes, smooth edges while preserving boundaries and recover fine edge details
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    # Extract text
    text = np.ones_like(Gray) * 255
    text[mask > 200] = Gray[mask > 200]

    return(character_recognition(text))
    #return text

# 4. Character recognition
def character_recognition(text):
    #grey = cv2.cvtColor(text, cv2.COLOR_BGR2GRAY)
    text_inv = cv2.bitwise_not(text)
    _, thresh = cv2.threshold(text_inv, 0, 255, cv2.THRESH_BINARY)
    thresh_inv = cv2.bitwise_not(thresh)
    scale = 3
    thresh_scaled = cv2.resize(thresh_inv, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    # Save temporary image for EasyOCR
    temp_path = "temp_ocr_input.png"
    cv2.imwrite(temp_path, thresh_scaled)
    
    # Initialize EasyOCR reader (downloads model on first run)
    reader = ocr.Reader(['en'], gpu=True)  # Set gpu=True if you have CUDA
    
    # Perform OCR
    results = reader.readtext(temp_path, detail=1, paragraph=False)
    
    # Extract and print text
    print("EasyOCR Results:")
    print("-" * 50)
    for detection in results:
        bbox, text, confidence = detection
        print(f"Text: {text}")
        print(f"Confidence: {confidence:.2f}")
        print("-" * 50)
    

    return thresh_scaled



# 1. Load image (replace with your handwritten image)
#f = eg.fileopenbox(msg="Choose a file to open", title="Open Image", filetypes=["*.jpg","*.jpeg","*.png", "*.tiff"])
img = cv2.imread("C:\\Users\\Wirexia\\Documents\\GitHub\\Script2Text\\Images\\Sample1.jpg")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # BGR is weird

cv2.namedWindow("Image", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Image", 1000, 1000)
cv2.imshow("Image", pre_processing(img))

cv2.waitKey(0)
cv2.destroyAllWindows()

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

C:\Python313\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (104332671 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


KeyboardInterrupt: 